In [155]:
import av
from av import VideoFrame
from linux_playground.utils.path_utils.specific_paths import relative_path

In [156]:
# paths and configs
file_prev_name = 'output.mp4'
file_next_name: str = 'output_for_whatsapp.mp4'
file_temp_media = 'tmp.png'
specific_path = fr'/home/armando/git_repos/linux_playground/linux_playground/python_playground/python_camera/python_cctv/'
video_file_path = fr'{specific_path}{file_prev_name}'
video_file_output_generate_path = fr'{specific_path}{file_next_name}'

from pathlib import Path
video_frame_output_path_png = Path(specific_path) / 'temp_media'

In [157]:

# new container 28_02_2026
container = av.open(video_file_path)
stream = container.streams.video[0]
fps = stream.average_rate.numerator
fps_float: float = float(fps)
container, stream, fps, fps_float

(<av.InputContainer '/home/armando/git_repos/linux_playground/linux_playground/python_playground/python_camera/python_cctv/output.mp4'>,
 <av.VideoStream #0 mpeg4, yuv420p 640x480 at 0x7940aac1bec0>,
 30,
 30.0)

In [158]:
for index, video_frame in enumerate(container.decode(stream)):
    if index % fps_float == 0:
        video_frame.to_image().save(video_frame_output_path_png / f'frame-{index}.png')

In [159]:
# Video Saving paths
video_file_path, video_file_output_generate_path

('/home/armando/git_repos/linux_playground/linux_playground/python_playground/python_camera/python_cctv/output.mp4',
 '/home/armando/git_repos/linux_playground/linux_playground/python_playground/python_camera/python_cctv/output_for_whatsapp.mp4')

In [160]:
# Input and Output Containers
input_container = av.open(video_file_path)
output_container = av.open(video_file_output_generate_path, mode='w')
input_container, output_container, input_container.format.long_name


(<av.InputContainer '/home/armando/git_repos/linux_playground/linux_playground/python_playground/python_camera/python_cctv/output.mp4'>,
 <av.OutputContainer '/home/armando/git_repos/linux_playground/linux_playground/python_playground/python_camera/python_cctv/output_for_whatsapp.mp4'>,
 'QuickTime / MOV')

In [161]:
# Make input & output streams
input_stream = input_container.streams.video[0]
# make it the same stream as template of the input_stream

codec_name = input_stream.codec_context.name
# output_stream = output_container.add_stream(codec_name)
whatsapp_acceptable_code_name = "libx264"
# Only ONE output stream
output_stream = output_container.add_stream(whatsapp_acceptable_code_name, rate=input_stream.average_rate)
output_stream.width = input_stream.codec_context.width
output_stream.height = input_stream.codec_context.height
output_stream.pix_fmt = 'yuv420p'
output_stream.time_base = input_stream.time_base

In [162]:
for frame in input_container.decode(input_stream):
    # encode the frame to the output stream
    packet = output_stream.encode(frame)
    if packet:
        output_container.mux(packet)

# flush encoder
for packet in output_stream.encode():
    output_container.mux(packet)

input_container.close()
output_container.close()


In [163]:
# for packet in input_container.demux(input_stream):
#     if packet.dts is None:
#         continue
#     # packet.stream = output_stream
#     output_container.mux(packet)
# input_container.close()
# output_container.close()
